# OpenAI Retry And Merge

Use this notebook for two recovery workflows:

1. Retry failed OpenAI batch requests by rebuilding a prompt file from failed `custom_id`s.
2. Merge critiques results from `1000_tasks`, `all_tasks`, and optional retry response files into one final parquet.

## Imports

In [ ]:
import sys
import json
import shutil
from pathlib import Path
import pandas as pd

sys.path.append('../..')

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.data_setup import prepare_project_data, merge_subset_into_full_by_key
from utils.models_setup import setup_client
from utils.openai_batch_manager import OpenAIBatchManager


## Config

In [ ]:
STAGE = "critiques"             # this notebook is mainly intended for critiques recovery/merge
PROMPTING_TYPE = "zero_shot"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1
PROVIDER = "openai"
MODEL_SHORT_OVERRIDE = None      # optional: set if you want to override cfg["model_short"]

BATCH_TASK_SUBSET = "all_tasks"  # the original batch run whose failures you are retrying
RETRY_DIR_NAME = "Retry_Failed"
BASE_RESPONSE_PATTERN = "responses-batch_[12].jsonl"
RETRY_RESPONSE_PATTERN = "responses-batch_*-retry.jsonl"

SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE if STAGE == "critiques" else RATING_SYSTEM_MESSAGE
build_prompt_fn = build_critique_prompt if STAGE == "critiques" else build_rating_prompt
update_results_df_fn = update_critiques_in_df if STAGE == "critiques" else update_rating_in_df

client, cfg = setup_client(
    provider=PROVIDER,
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

model_short = MODEL_SHORT_OVERRIDE or cfg["model_short"]
cfg


## Paths

In [ ]:
batch_main_dir = Path(f"./{STAGE}_Batching/Batch_Files-{BATCH_TASK_SUBSET}-{PROMPTING_TYPE}-{PROVIDER}")
responses_dir = batch_main_dir / "Batch_Responses"
error_dir = batch_main_dir / "Batch_Errors"
retry_root = batch_main_dir / RETRY_DIR_NAME
retry_prompts_dir = retry_root / "Batch_Prompts"
retry_responses_dir = retry_root / "Batch_Responses"

retry_root.mkdir(parents=True, exist_ok=True)
retry_prompts_dir.mkdir(parents=True, exist_ok=True)
retry_responses_dir.mkdir(parents=True, exist_ok=True)
error_dir.mkdir(parents=True, exist_ok=True)

prompts_file = batch_main_dir / f"prompts-{BATCH_TASK_SUBSET}.jsonl"
if not prompts_file.exists():
    prompt_candidates = sorted(batch_main_dir.glob("prompts-*.jsonl"))
    if prompt_candidates:
        prompts_file = prompt_candidates[0]

print("batch_main_dir:", batch_main_dir)
print("prompts_file:", prompts_file)
print("responses_dir:", responses_dir)
print("retry_root:", retry_root)


## Batch Manager Setup

In [ ]:
batch_process = OpenAIBatchManager(
    client=client,
    batch_main_dir=batch_main_dir,
    selected_tasks=BATCH_TASK_SUBSET,
)

retry_batch = OpenAIBatchManager(
    client=client,
    batch_main_dir=retry_root,
    selected_tasks="retry_failed",
)

batch_process.prompts_file = prompts_file
batch_process.responses_dir = responses_dir
retry_batch.responses_dir = retry_responses_dir

batch_process, retry_batch


## Inspect Existing Batches

In [ ]:
batches = batch_process.list_batches(limit=200)


In [ ]:
rows = []
if batches:
    for b in batches:
        input_file_id = getattr(b, "input_file_id", None)
        input_name = batch_process._retrieve_file_name(input_file_id) if input_file_id else None
        if input_name and input_name.startswith("prompts-batch_"):
            rc = getattr(b, "request_counts", None)
            rows.append({
                "batch_id": b.id,
                "status": b.status,
                "input_name": input_name,
                "completed": getattr(rc, "completed", None),
                "failed": getattr(rc, "failed", None),
                "total": getattr(rc, "total", None),
                "output_file_id": getattr(b, "output_file_id", None),
                "error_file_id": getattr(b, "error_file_id", None),
            })

batches_df = pd.DataFrame(rows).sort_values("input_name") if rows else pd.DataFrame()
batches_df


## Download Error Files

In [ ]:
for _, row in batches_df.iterrows():
    error_file_id = row.get("error_file_id")
    if error_file_id:
        out = error_dir / f"errors-{row['input_name'].replace('prompts-', '').replace('.jsonl', '.jsonl')}"
        content = client.files.content(error_file_id)
        content.write_to_file(out)
        print("saved:", out)


## Collect Failed Requests

In [ ]:
error_files = sorted(error_dir.glob("errors-*.jsonl"))
failed_records = []
for p in error_files:
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                failed_records.append(json.loads(line))

print("failed records:", len(failed_records))
failed_records[:3]


In [ ]:
def extract_error_code(rec):
    return rec.get("response", {}).get("body", {}).get("error", {}).get("code")

def extract_error_message(rec):
    return rec.get("response", {}).get("body", {}).get("error", {}).get("message")

failed_df = pd.DataFrame({
    "custom_id": [r.get("custom_id") for r in failed_records],
    "error_code": [extract_error_code(r) for r in failed_records],
    "error_message": [extract_error_message(r) for r in failed_records],
})

failed_df["error_code"].value_counts(dropna=False)


## Build Retry Prompts

In [ ]:
FAILED_ERROR_CODE = "insufficient_quota"   # set to None if you want all failed custom_ids

if FAILED_ERROR_CODE is None:
    failed_custom_ids = sorted(set(failed_df["custom_id"].dropna()))
else:
    failed_custom_ids = sorted(set(failed_df.loc[failed_df["error_code"] == FAILED_ERROR_CODE, "custom_id"].dropna()))

print("failed custom_ids selected:", len(failed_custom_ids))
failed_custom_ids[:10]


In [ ]:
retry_prompts_file = retry_root / "prompts-retry-failed.jsonl"
retry_summary = batch_process.build_retry_prompts_from_failed_requests(
    failed_custom_ids,
    source_prompts_file=prompts_file,
    output_file=retry_prompts_file,
)

retry_summary


In [ ]:
with open(retry_prompts_file, "r", encoding="utf-8") as f:
    retry_prompt_records = [json.loads(line) for line in f if line.strip()]

print("retry lines:", len(retry_prompt_records))
print("unique retry ids:", len({r["custom_id"] for r in retry_prompt_records}))


## Run Retry Batch

In [ ]:
retry_batch.prompts_file = retry_prompts_file
retry_batch.prompts_dir = retry_prompts_dir
retry_batch.responses_dir = retry_responses_dir
retry_batch.prompts_dir.mkdir(parents=True, exist_ok=True)
retry_batch.responses_dir.mkdir(parents=True, exist_ok=True)

retry_batch.split_prompts_jsonl(max_lines_per_file=500)


In [ ]:
# Automatic retry flow
start_from_batch = 1
resume_batch = None

retry_batch.process_batches_from(
    start_from_batch=start_from_batch,
    sleep_minutes=2,
    resume_batch_id=resume_batch,
)


In [ ]:
# Optional short-path download workaround for Windows path issues
TEMP_RETRY_DIR = Path(r"C:\temp\oa_retry_responses")
TEMP_RETRY_DIR.mkdir(parents=True, exist_ok=True)

# Example usage:
# retry_batch.responses_dir = TEMP_RETRY_DIR
# retry_batch.batch_id = "batch_XXXXXXXXXXXXXXXXXXXXXXXX"
# state, batch_obj = retry_batch.check_batch(verbose=True)
# out_path = retry_batch.retrieve_batch_output(batch_obj=batch_obj, base_name="responses")


## Move Downloaded Retry Files Into Project (Optional)

In [ ]:
temp_retry_dir = Path(r"C:\temp\oa_retry_responses")
if temp_retry_dir.exists():
    for p in sorted(temp_retry_dir.glob("responses*.jsonl")):
        dst = responses_dir / p.name.replace(".jsonl", "-retry.jsonl")
        shutil.copy2(p, dst)
        print("copied:", p, "->", dst)
else:
    print("No temp retry dir found:", temp_retry_dir)


## Merge All Critiques Results

In [ ]:
if STAGE != "critiques":
    raise ValueError("This merge flow is configured for critiques runs.")

results_1000_file = Path(f"../../results/{STAGE}/parquet/{STAGE}-{PROMPTING_TYPE}-1000_tasks-{model_short}-responses.parquet")
print("1000 parquet:", results_1000_file)


In [ ]:
responses_df_all_fresh, results_paths = prepare_project_data(
    model_short=model_short,
    num_trials=NUM_TRIALS,
    task_subset="all_tasks",
    shots=PROMPTING_TYPE,
    stage=STAGE,
)

final_results_file = results_paths["results_parquet"]
print("fresh all_tasks rows:", len(responses_df_all_fresh))


In [ ]:
summary_all = batch_process.extract_results_from_responses(
    responses_df_all_fresh,
    update_results_df_fn,
    pattern=BASE_RESPONSE_PATTERN,
)

print(
    "all_tasks summary:",
    summary_all["updated"], "updated |",
    len(summary_all["skipped"]), "skipped |",
    len(summary_all["errors"]), "errors",
)

retry_source_dir = responses_dir
retry_matches = sorted(retry_source_dir.glob(RETRY_RESPONSE_PATTERN))
if retry_matches:
    summary_retry = batch_process.extract_results_from_responses(
        responses_df_all_fresh,
        update_results_df_fn,
        pattern=RETRY_RESPONSE_PATTERN,
    )
    print(
        "retry summary:",
        summary_retry["updated"], "updated |",
        len(summary_retry["skipped"]), "skipped |",
        len(summary_retry["errors"]), "errors",
    )
else:
    print("No retry files matched:", RETRY_RESPONSE_PATTERN)


In [ ]:
responses_df_1000 = pd.read_parquet(results_1000_file)
responses_df_merged, diag = merge_subset_into_full_by_key(
    full_df=responses_df_all_fresh,
    subset_df=responses_df_1000,
    key="screen_task_id",
    value_col="critiques",
    only_fill_missing=True,
)

diag


In [ ]:
missing_final = (responses_df_merged["critiques"].isna() | (responses_df_merged["critiques"] == "")).sum()

print("final rows:", len(responses_df_merged))
print("final unique screen_task_id:", responses_df_merged["screen_task_id"].nunique())
print("final missing critiques:", missing_final)

responses_df_merged.head(2)


In [ ]:
responses_df_merged.to_parquet(final_results_file, index=False)
print("Saved merged results:", final_results_file)
